# 验证期选择训练轮次
2011–2019训练、2020–2022验证、2023后复用诊断。只选择训练轨迹中的checkpoint，不重拟合验证期；与同轨迹末轮和长期强参考比较。

In [1]:
from pathlib import Path
from dataclasses import asdict
from copy import deepcopy
import os,json,hashlib
import numpy as np,pandas as pd,torch,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _dataset,_auxiliary_dataset,_prediction_dates,_ensemble
from src.model.minute_pit_allocator import TARGETS
from src.model.validation_epoch_allocator import ValidationEpochAllocator
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/validation_epoch_training_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==p['release_sha256']
training,ta=_source(root,p['training_source'],['inputs/daily_frame','inputs/features','inputs/common','inputs/timing','inputs/auxiliary','inputs/audit',*[f'models/base_2020_{seed}' for seed in p['seeds']]])
source,sa=_source(root,p['source'],['inputs/frame']);controller,ca=_source(root,p['price_source'],['inputs/controller','inputs/plan']);action,aa=_source(root,p['action_source'],['inputs/plan','inputs/market','controller/source_recovery_selection']);strong,sta=_source(root,p['strong_source'],['inputs/plan',*[f'analysis/native/{family}_{cost}' for family in ['candidate','control','soxx'] for cost in p['slippage_bps']]])
for key in ['initial_cash','tariff','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert strong['inputs/plan'][key]==p[key]
assert controller['inputs/plan']['tariff']==p['tariff']
original=training['models/base_2020_71'];base=training['inputs/daily_frame'];dataset=_dataset(training['inputs/features'],base[list(TARGETS)],training['inputs/common'],training['inputs/timing'],{'train':(original.spec.train_start,str(original.train_end.date())),'valid':(p['validation_start'],p['validation_end']),'predict':(p['start'],p['end'])});auxiliary=_auxiliary_dataset(training['inputs/auxiliary']['2020'],str(original.train_end.date()))
state_path=card/'run_state.json';digest=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest();state=json.loads(state_path.read_text()) if state_path.exists() else {'plan_sha256':digest,'fits':[],'native':[]};assert state['plan_sha256']==digest
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if state.get('recorder_id'):kwargs.update(recorder_id=state['recorder_id'],resume=True)
predictions={};audits=[];selection=[]
with R.start(**kwargs):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(state,state_path);rec.save_objects(artifact_path='inputs',plan=p,source_audits={'training':ta,'source':sa,'controller':ca,'action':aa,'strong':sta});rec.save_objects(artifact_path='sources',model=(root/'src/model/validation_epoch_allocator.py').read_text(),base_model=(root/'src/model/minute_pit_allocator.py').read_text())
 for seed in p['seeds']:
  template=training[f'models/base_2020_{seed}'];name=f'trajectory_{seed}'
  if name in state['fits']:model=rec.load_object('models/'+name)
  else:
   assert len(state['fits'])<p['maximum_model_fits'];model=ValidationEpochAllocator(base_features=template.base_features,extra_features=[],feature_masks=template.feature_masks,spec={**asdict(template.spec),'root':str(root)},seed=template.seed,cutoff=str(template.cutoff),train_end=str(template.train_end.date()),minimum_observations=template.minimum_observations);model.validation_cutoff=p['validation_cutoff'];model.fit(dataset,auxiliary_dataset=auxiliary);rec.save_objects(artifact_path='models',**{name:model});state['fits'].append(name);atomic_json(state,state_path)
  for key,value in template.network.state_dict().items():torch.testing.assert_close(value,model.network.state_dict()[key],rtol=0,atol=0)
  assert model.primary_scaler==template.primary_scaler and model.common_scaler==template.common_scaler
  predictions[('control',seed)]=_prediction_dates(model.predict(dataset));candidate=deepcopy(model);candidate.network.load_state_dict(candidate.selected_state);candidate.audit={**candidate.audit,'checkpoint_policy':'minimum_fixed_validation_loss','selected_epoch':candidate.selected_epoch};predictions[('candidate',seed)]=_prediction_dates(candidate.predict(dataset));rec.save_objects(artifact_path='models',**{f'selected_{seed}':candidate});audits.append({'seed':seed,**model.audit});selection.append({'seed':seed,'selected_epoch':model.selected_epoch,'selected_loss':model.best_validation_loss,'final_loss':model.validation_history[-1]['loss'],'frozen_final_state_identical':True,**model.validation_audit});rec.log_metrics(completed_fits=len(state['fits']))
 rec.save_objects(artifact_path='analysis',predictions=predictions,training=audits,selection=selection)
frame=source['inputs/frame'];frame=frame.loc[frame.signal_date.between(p['start'],p['end'])].copy();market=action['inputs/market'];dates=pd.DatetimeIndex(market.signal_date);returns=market.set_index('signal_date')[['soxl_return','soxx_return']];targets={f:_smooth_weights(_ensemble(predictions,f,p['seeds']).reindex(dates),spec=original.spec,candidate=original.candidate,asset_returns=returns) for f in ['control','candidate']}
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);books={}
for family in ['control','candidate']:
 local=frame.copy();aligned=targets[family].reindex(pd.DatetimeIndex(frame.signal_date));assert aligned.notna().all().all()
 for col in aligned:local[col]=aligned[col].to_numpy()
 for cost in p['slippage_bps']:
  key=f'{family}_{cost}'
  if key in state['native']:book=rec.load_object('analysis/native/'+key)
  else:
   assert len(state['native'])<p['maximum_new_native_cases'];replay=replay_full_path(local,models=None,plan=p,slippage_bps=cost,controller=controller['inputs/controller'],recovery_spec=recovery,action_spec=spec,selection=action['controller/source_recovery_selection']);book=native_tiered_stage1(local,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=cost,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets'];rec.save_objects(artifact_path='analysis/native',**{key:book});state['native'].append(key);atomic_json(state,state_path)
  books[(family,cost)]=book;rec.log_metrics(completed_native_cases=len(state['native']))
for cost in p['slippage_bps']:
 for family,key in [('full_history_tangent','candidate'),('full_history_original','control'),('soxx','soxx')]:books[(family,cost)]=strong[f'analysis/native/{key}_{cost}'];rec.save_objects(artifact_path='analysis/native',**{f'{family}_{cost}':books[(family,cost)]})
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];b=books[('soxx',cost)]['authority'];assert path.trade_date.equals(b.trade_date);rel=np.log1p(path.strategy_return)-np.log1p(b.strategy_return);rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,b),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'average_cash_pct':100*path.ending_cash_weight.mean(),'positive_126d_fraction':rel.rolling(126).sum().dropna().gt(0).mean()});annual.append(annual_paired_metrics(path,b).assign(family=family,slippage_bps=cost))
 if family=='candidate':
  for comparator in ['control','soxx','full_history_tangent']:uncertainty.append({'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);uncertainty=pd.DataFrame(uncertainty);cnd=table[table.family.eq('candidate')].set_index('slippage_bps');ctl=table[table.family.eq('control')].set_index('slippage_bps');stronger=table[table.family.eq('full_history_tangent')].set_index('slippage_bps');gates={'beat_paired_control_all_costs':bool(cnd.terminal_wealth.gt(ctl.terminal_wealth+1e-8).all()),'beat_strong_all_costs':bool(cnd.terminal_wealth.gt(stronger.terminal_wealth+1e-8).all()),'net_soxx_excess_all_costs':bool(cnd.relative_wealth.gt(1).all()),'drawdown_30pct':bool(cnd.max_drawdown_pct.ge(-30).all() and cnd.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(cnd.cvar5_pct.ge(cnd.benchmark_cvar5_pct).all()),'rolling_70pct':bool(cnd.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq('candidate')].groupby('slippage_bps').relative_wealth.apply(lambda x:x.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty.lower_bound_positive.all()),'independent_validation':False}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','sources/*','analysis/predictions','analysis/native/*']};summary={'experiment':card.name,'decision':'needs_independent_validation' if all(v for k,v in gates.items() if k!='independent_validation') else p['failure_decision'],'gates':gates,'new_fits':len(state['fits']),'new_native_cases':len(state['native']),'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False,'plan_sha256':digest};rec.save_objects(artifact_path='analysis',summary=summary,comparison=table,annual=annual,uncertainty=uncertainty,targets=targets);rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in (Path(summary['recorder_local_dir'])/'artifacts').rglob('*')))
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty),('training_audit',pd.DataFrame(audits)),('epoch_selection',pd.DataFrame(selection))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'segments':dataset.segments,'artifact_policy':policy}));a=Path(summary['recorder_local_dir'])/'artifacts';atomic_json({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');print(pd.DataFrame(selection).to_string(index=False));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct']].to_string(index=False))


# 验证期选择轮次：配对改善，仍不晋升

完成3条新训练轨迹（各300轮）和6套新增Qlib原生完整账户。训练为2011-01-03至2019-12-02，2244条成熟主样本，历史资格限制下辅助样本0条；两组完全一致。验证为2020-01-02至2022-12-21的750条成熟标签，最晚成熟2022-12-30 09:31，美东2023-01-01以前可用；训练期拟合缩放器，验证期不更新参数。训练结束与验证开始保留原有purge间隔。

固定在1至300轮中按原主目标（配置损失+原0.02分类损失，不含辅助项）选择验证损失最小的一轮；完全相同时取最早轮次。seed71/172/273分别选择277/300/267轮，验证loss分别0.021845/0.022403/0.022161，对应末轮0.021902/0.022403/0.022197。未改变训练窗口、指标或再选择seed；等权集成。不在验证期重拟合。模型选择本身仍可能过拟合750条相关观察，不能因时间切分就声称消除了过拟合。

每条新训练轨迹的第300轮参数与此前冻结的同口径2020模型逐项完全一致（零容差），缩放器也相同，证明验证观察未改变优化路径。对照是同轨迹末轮；候选是选中的历史checkpoint，不是另一组随机初始化。selected模型元数据已明确标注选择轮次，完整轨迹模型保留原末轮身份。

回测2023-01-03信号至2026-10-09退出、944区间，10万美元，固定IBKR Pro Tiered研究费率投影，独立0/2/5bp账户及完整开盘/恢复/风险规则。旧历史已反复使用，验证期与外层都不构成新鲜独立证据。

|账户|0bp净收益|2bp净收益|5bp净收益|
|---|---:|---:|---:|
|同2019训练末轮|101.20%|100.05%|67.80%|
|验证期选择轮次|297.67%|292.83%|277.45%|
|原全历史小配置头|342.52%|335.36%|348.06%|
|原全历史原结构|280.24%|269.16%|266.08%|
|SOXX|387.75%|387.73%|387.70%|

2bp候选监控回撤29.63%，同训练期末轮30.84%；回撤与CVaR门通过。滚动126日正超额占比34.43%，未达70%；净SOXX超额、年度稳定性、强参考及联合区块置信门失败。相对同训练期末轮的三个成本区块区间下界均为正，但该区间条件于复用路径，不包含模型选择与重训不确定性。不能只展示相对较弱同训练期对照的改善，忽略长期强参考和SOXX。

这支持“固定末轮不是该训练轨迹的最好验证选择”，但不证明全部现金问题由过拟合导致。验证损失差很小，完整策略收益差很大；模型差异可经阈值、恢复及持仓路径放大，不能把整个账户差额当成独立预测alpha。训练窗口及辅助覆盖与当前全历史模型不同，因此也不能直接替换生产或自动对全历史模型继续调轮次。

决定not_promoted_validation_epoch。关闭本固定训练停止协议，不改验证期、选择指标、patience、训练窗或epoch预算挽救。生产不变，稳定超额未完成。下一需要是能跨训练期重复的预测信息或泛化改善，而非再对同一复用区间选择停止参数。

Qlib模型/预测/完整账户与来源已NAS归档、冷恢复逐对象哈希通过；4项针对性测试通过，包含未成熟验证标签排除和训练/验证重叠拒绝。实际成交时延、全分钟回撤、真实派息及历史价格代理的既有边界保留。

2bp年度候选 / SOXX：2023：34.26% / 56.60%；2024：26.44% / 20.97%；2025：35.02% / 46.80%；2026：71.38% / 75.38%。2026为不完整年度。
